In [8]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

BASE_DIR = Path.cwd().parent

INPUT_FILE = BASE_DIR / "Milestone1" / "university_raw_data.csv"
OUTPUT_FILE = BASE_DIR / "Milestone1" / "university_cleaned.csv"

In [9]:
df = pd.read_csv(INPUT_FILE)

print("Initial shape:", df.shape)

Initial shape: (6517, 25)


In [10]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

print(df.columns.tolist())

['university_name', 'country', 'region', 'source', 'ranking_year', 'global_rank', 'overall_score', 'academic_reputation_score', 'employer_reputation_score', 'faculty_student_score', 'student_staff_ratio', 'citations_per_faculty_score', 'citations_score', 'teaching_score', 'research_score', 'industry_income_score', 'international_outlook_score', 'international_faculty_score', 'international_students_score', 'international_student_percentage', 'international_research_network_score', 'employment_outcomes_score', 'sustainability_score', 'number_of_students', 'female_male_ratio']


In [11]:
#clean university names
def normalize_university_name(name):
    if pd.isna(name):
        return pd.NA

    name = str(name).strip().lower()

    name = unicodedata.normalize("NFKD", name)
    name = "".join(
        c for c in name
        if not unicodedata.combining(c)
    )

    name = re.sub(r"[^\w\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name


df["university_name"] = (
    df["university_name"]
    .apply(normalize_university_name)
)

In [12]:
#standadize country names
COUNTRY_ALIASES = {
    "United States of America": "United States",
    "USA": "United States",
    "US": "United States",

    "UK": "United Kingdom",
    "Great Britain": "United Kingdom",

    "Russian Federation": "Russia",

    "Iran, Islamic Republic of": "Iran",

    "China (Mainland)": "China",

    "Hong Kong SAR": "Hong Kong",

    "Macau SAR": "Macau",

    "Brunei Darussalam": "Brunei",

    "Palestinian Territory, Occupied": "Palestine",
}

df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
    .replace(COUNTRY_ALIASES)
)

In [13]:
# Fill country from a matching university record when safely available
country_from_match = (
    df.groupby("university_name")["country"]
    .transform(
        lambda x: x.dropna().iloc[0]
        if x.notna().any()
        else pd.NA
    )
)

df["country"] = df["country"].fillna(country_from_match)

#Known recoverable country
df.loc[
    df["university_name"].eq("southwest jiaotong university"),
    "country"
] = df.loc[
    df["university_name"].eq("southwest jiaotong university"),
    "country"
].fillna("China")

In [14]:
# Remove exact duplicate
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

df = df.drop_duplicates().copy()

print("Shape after duplicate removal:", df.shape)

Duplicate rows: 29
Shape after duplicate removal: (6488, 25)


In [15]:
#clean percentage and ratio field

#international student percentage
if "international_student_percentage" in df.columns:
    df["international_student_percentage"] = (
        df["international_student_percentage"]
        .astype("string")
        .str.replace("%", "", regex=False)
        .str.strip()
    )

    df["international_student_percentage"] = pd.to_numeric(
        df["international_student_percentage"],
        errors="coerce"
    )

#female/male ratio->female percentage
if "female_male_ratio" in df.columns:

    ratio = (
        df["female_male_ratio"]
        .astype("string")
        .str.replace(" ", "", regex=False)
    )

    df["female_percentage"] = pd.to_numeric(
        ratio.str.split(":", expand=True)[0],
        errors="coerce"
    )

    df = df.drop(columns=["female_male_ratio"])

In [16]:
#conver numeric fields
numeric_columns = [
    "ranking_year",
    "overall_score",
    "academic_reputation_score",
    "employer_reputation_score",
    "faculty_student_score",
    "student_staff_ratio",
    "citations_per_faculty_score",
    "citations_score",
    "teaching_score",
    "research_score",
    "industry_income_score",
    "international_outlook_score",
    "international_faculty_score",
    "international_students_score",
    "international_research_network_score",
    "employment_outcomes_score",
    "sustainability_score",
    "number_of_students",
    "female_percentage"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = (
            df[column]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.replace("%", "", regex=False)
        )

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

In [17]:
# Create university matching key
df["university_match_key"] = df["university_name"]

UNIVERSITY_ALIASES = {
    "the university of melbourne": "university of melbourne",
    "the university of sydney": "university of sydney",
    "the university of queensland": "university of queensland",
    "the university of western australia": "university of western australia",
}

df["university_match_key"] = (
    df["university_match_key"]
    .replace(UNIVERSITY_ALIASES)
)

# Create university ID
university_master = (
    df[["university_match_key"]]
    .dropna()
    .drop_duplicates()
    .sort_values("university_match_key")
    .reset_index(drop=True)
)

university_master["university_id"] = [
    f"U{i:04d}"
    for i in range(1, len(university_master) + 1)
]

university_id_map = dict(
    zip(
        university_master["university_match_key"],
        university_master["university_id"]
    )
)

df["university_id"] = (
    df["university_match_key"]
    .map(university_id_map)
)

print("Unique universities:", df["university_id"].nunique())
print(
    "Missing university IDs:",
    df["university_id"].isna().sum()
)


Unique universities: 3468
Missing university IDs: 79


In [18]:
#Create country ID
country_master = (
    df[["country"]]
    .dropna()
    .drop_duplicates()
    .sort_values("country")
    .reset_index(drop=True)
)

country_master["country_id"] = [
    f"C{i:03d}"
    for i in range(1, len(country_master) + 1)
]

country_id_map = dict(
    zip(
        country_master["country"],
        country_master["country_id"]
    )
)

df["country_id"] = df["country"].map(country_id_map)

print("Unique countries:", df["country"].nunique())
print("Unique country IDs:", df["country_id"].nunique())
print("Missing country IDs:", df["country_id"].isna().sum())


Unique countries: 134
Unique country IDs: 134
Missing country IDs: 79


In [19]:
# Inspect rank formats before normalization
for source in ["QS", "THE", "WUR"]:
    ranks = (
        df.loc[df["source"] == source, "global_rank"]
        .dropna()
        .astype(str)
        .drop_duplicates()
    )

    non_numeric = ranks[
        ~ranks.str.fullmatch(r"\d+(?:\.\d+)?")
    ]

    print(f"\n{source}")
    print("Unique rank values:", len(ranks))
    print("Non-numeric/range values:", non_numeric.tolist()[:30])



QS
Unique rank values: 388
Non-numeric/range values: ['621-630', '601-610', '611-620', '631-640', '641-650', '651-660', '661-670', '671-680', '681-690', '691-700', '701-710', '711-720', '721-730', '731-740', '741-750', '751-760', '761-770', '771-780', '781-790', '791-800', '801-850', '851-900', '901-950', '951-1000', '1001-1200', '1201-1400', '1401+']

THE
Unique rank values: 160
Non-numeric/range values: ['=30', '=38', '=55', '=64', '=87', '=92', '=95', '=97', '=99', '=103', '=106', '=109', '=111', '=116', '=119', '=123', '=130', '=136', '=138', '=140', '=143', '=145', '=150', '=152', '=155', '=158', '=161', '=164', '=168', '=175']

WUR
Unique rank values: 162
Non-numeric/range values: ['201–250', '251–300', '301–350', '351–400', '401–500', '501–600', '601–800', '801–1000', '1001–1200', '1201–1500', '1501+', 'Reporter', '-']


In [20]:
#Normalize ranking
def normalize_rank(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value in {"", "-", "Reporter"}:
        return np.nan

    value = value.replace("–", "-")
    value = value.replace("=", "")

    if value.endswith("+"):
        try:
            return float(value[:-1])
        except ValueError:
            return np.nan

    if "-" in value:
        parts = value.split("-")

        if len(parts) == 2:
            try:
                low = float(parts[0])
                high = float(parts[1])
                return (low + high) / 2
            except ValueError:
                return np.nan

    try:
        return float(value)

    except ValueError:
        return np.nan


df["normalized_rank"] = (
    df["global_rank"]
    .apply(normalize_rank)
)

In [21]:
#Check ranking normalization
print(
    df[
        ["global_rank", "normalized_rank"]
    ].head(20)
)

print(
    "\nMissing normalized ranks:",
    df["normalized_rank"].isna().sum()
)

print(
    "\nRank records by source:"
)

print(
    df.groupby("source")["normalized_rank"]
    .apply(lambda x: x.notna().sum())
)

   global_rank  normalized_rank
0            1              1.0
1            2              2.0
2            3              3.0
3            4              4.0
4            5              5.0
5            6              6.0
6            7              7.0
7            8              8.0
8            9              9.0
9           10             10.0
10          11             11.0
11          12             12.0
12          13             13.0
13          14             14.0
14          15             15.0
15          16             16.0
16          17             17.0
17          18             18.0
18          19             19.0
19          20             20.0

Missing normalized ranks: 1384

Rank records by source:
source
QS     1503
THE    1904
WUR    1697
Name: normalized_rank, dtype: int64


In [22]:
#Identufy unranked/reporting records
unranked = df[df["normalized_rank"].isna()].copy()

print("Unranked/unusable rank records:", len(unranked))
print("\nBy source:")
print(unranked["source"].value_counts())

print("\nOriginal global_rank values:")
print(unranked["global_rank"].value_counts(dropna=False))

Unranked/unusable rank records: 1384

By source:
source
THE    769
WUR    615
Name: count, dtype: int64

Original global_rank values:
global_rank
Reporter    1281
-            103
Name: count, dtype: int64


In [23]:
# Create Tableau-ready ranked dataset
cleaned_df = df[
    df["normalized_rank"].notna()
].copy()

max_rank = cleaned_df["normalized_rank"].max()

cleaned_df["ranking_score"] = (
    100 * (
        1 -
        (cleaned_df["normalized_rank"] - 1) /
        (max_rank - 1)
    )
).round(2)


def get_rank_band(rank):
    if pd.isna(rank):
        return pd.NA
    if rank <= 10:
        return "1-10"
    elif rank <= 50:
        return "11-50"
    elif rank <= 100:
        return "51-100"
    elif rank <= 250:
        return "101-250"
    elif rank <= 500:
        return "251-500"
    elif rank <= 1000:
        return "501-1000"
    elif rank <= 1500:
        return "1001-1500"
    else:
        return "1501+"


cleaned_df["rank_band"] = (
    cleaned_df["normalized_rank"]
    .apply(get_rank_band)
)

final_columns = [
    "university_id",
    "university_name",
    "country_id",
    "country",
    "region",
    "source",
    "ranking_year",
    "global_rank",
    "normalized_rank",
    "ranking_score",
    "rank_band",
    "overall_score",
    "academic_reputation_score",
    "employer_reputation_score",
    "faculty_student_score",
    "student_staff_ratio",
    "citations_per_faculty_score",
    "citations_score",
    "teaching_score",
    "research_score",
    "industry_income_score",
    "international_outlook_score",
    "international_faculty_score",
    "international_students_score",
    "international_student_percentage",
    "international_research_network_score",
    "employment_outcomes_score",
    "sustainability_score",
    "number_of_students",
    "female_percentage",
]

final_columns = [
    column
    for column in final_columns
    if column in cleaned_df.columns
]

cleaned_df = cleaned_df[final_columns].copy()

print("Rows before:", len(df))
print("Rows after:", len(cleaned_df))
print("Rows excluded as unranked:", len(df) - len(cleaned_df))


Rows before: 6488
Rows after: 5104
Rows excluded as unranked: 1384


In [24]:
#Validate duplicates and critical iddentifiers
print("Duplicate rows:", cleaned_df.duplicated().sum())
critical_columns = [
    "university_id",
    "university_name",
    "country_id",
    "country",
]

print(
    cleaned_df[critical_columns]
    .isna()
    .sum()
)

Duplicate rows: 0
university_id      0
university_name    0
country_id         0
country            0
dtype: int64


In [25]:
# Validate ranking fields and report performance-indicator missingness
core_columns = [
    "university_id",
    "university_name",
    "country_id",
    "country",
    "source",
    "ranking_year",
    "global_rank",
    "normalized_rank",
    "ranking_score",
    "rank_band",
]

core_missing = (
    cleaned_df[core_columns]
    .isna()
    .sum()
    .sum()
)

core_total = cleaned_df[core_columns].size

core_missing_percentage = (
    core_missing / core_total * 100
)

print("Core missing cells:", core_missing)
print(
    "Core missing percentage:",
    round(core_missing_percentage, 2),
    "%"
)

performance_columns = [
    "overall_score",
    "academic_reputation_score",
    "research_score",
    "citations_score",
    "teaching_score",
    "student_staff_ratio",
    "international_student_percentage",
    "number_of_students",
]

performance_columns = [
    column
    for column in performance_columns
    if column in cleaned_df.columns
]

print("\nPerformance indicator missingness (%):")
print(
    cleaned_df[performance_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)


Core missing cells: 0
Core missing percentage: 0.0 %

Performance indicator missingness (%):
overall_score                       80.41
academic_reputation_score           70.55
research_score                      29.45
citations_score                     29.45
teaching_score                      29.45
student_staff_ratio                 29.45
international_student_percentage    29.47
number_of_students                  29.45
dtype: float64


In [26]:
print("=" * 60)
print("MODULE 2 - FINAL VALIDATION")
print("=" * 60)

print("Rows:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))
print(
    "Unique universities:",
    cleaned_df["university_id"].nunique()
)
print(
    "Unique countries:",
    cleaned_df["country_id"].nunique()
)
print(
    "Duplicate rows:",
    cleaned_df.duplicated().sum()
)
print(
    "Core missing percentage:",
    round(core_missing_percentage, 2),
    "%"
)

print("\nSource distribution:")
print(cleaned_df["source"].value_counts())

print("\nRank band distribution:")
print(cleaned_df["rank_band"].value_counts())

MODULE 2 - FINAL VALIDATION
Rows: 5104
Columns: 30
Unique universities: 2657
Unique countries: 121
Duplicate rows: 0
Core missing percentage: 0.0 %

Source distribution:
source
THE    1904
WUR    1697
QS     1503
Name: count, dtype: int64

Rank band distribution:
rank_band
1001-1500    1491
501-1000     1465
251-500       752
1501+         648
101-250       448
51-100        148
11-50         122
1-10           30
Name: count, dtype: int64


In [27]:
cleaned_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved: {OUTPUT_FILE}")

Saved: c:\Users\Sweta Mondal\OneDrive\Desktop\EduVision_DV\Milestone1\university_cleaned.csv


In [28]:
# Verify exported dataset
exported_df = pd.read_csv(OUTPUT_FILE)

print("Exported shape:", exported_df.shape)
print("Exported duplicate rows:", exported_df.duplicated().sum())
print("\nExported source distribution:")
print(exported_df["source"].value_counts())

Exported shape: (5104, 30)
Exported duplicate rows: 0

Exported source distribution:
source
THE    1904
WUR    1697
QS     1503
Name: count, dtype: int64


In [29]:
# Preview final cleaned dataset
cleaned_df.head()

,university_id,university_name,country_id,country,region,source,ranking_year,global_rank,normalized_rank,ranking_score,...,industry_income_score,international_outlook_score,international_faculty_score,international_students_score,international_student_percentage,international_research_network_score,employment_outcomes_score,sustainability_score,number_of_students,female_percentage
0,U1270,massachusetts institute of technology mit,C127,United States,Americas,QS,2025,1,1.0,100.00,...,<NA>,<NA>,99.3,86.8,<NA>,96.0,100.0,99.0,<NA>,<NA>
1,U0835,imperial college london,C126,United Kingdom,Europe,QS,2025,2,2.0,99.93,...,<NA>,<NA>,100.0,99.6,<NA>,97.4,93.4,99.7,<NA>,<NA>
2,U3055,university of oxford,C126,United Kingdom,Europe,QS,2025,3,3.0,99.87,...,<NA>,<NA>,98.1,97.7,<NA>,100.0,100.0,85.0,<NA>,<NA>
3,U0759,harvard university,C127,United States,Americas,QS,2025,4,4.0,99.80,...,<NA>,<NA>,74.1,69.0,<NA>,99.6,100.0,84.4,<NA>,<NA>
4,U2732,university of cambridge,C126,United Kingdom,Europe,QS,2025,5,5.0,99.73,...,<NA>,<NA>,100.0,94.8,<NA>,99.3,100.0,84.8,<NA>,<NA>


In [30]:
missing_rank = df[df["global_rank"].isna()]

print("Missing global_rank:", len(missing_rank))

print(
    missing_rank[
        [
            "university_id",
            "university_name",
            "country",
            "source",
            "ranking_year",
            "global_rank",
            "normalized_rank"
        ]
    ].to_string(index=False)
)

Missing global_rank: 0
Empty DataFrame
Columns: [university_id, university_name, country, source, ranking_year, global_rank, normalized_rank]
Index: []
